# Hardware Pioneers Exhibitor Link Extractor + Keyword Ranking

Use this notebook when:
- links appear in **Inspect Element** but not **View Source**
- you copied rendered HTML from DevTools
- you want company/exhibitor URLs ranked by relevance to FPGA, embedded, semiconductor, IoT, Edge AI, etc.

Fast workflow:
1. Paste copied HTML into `html_text`
2. Run extraction
3. Run ranking
4. Export CSV

In [ ]:
# Optional install cell
# Run this only if imports fail.
# In Jupyter/VS Code notebooks this is okay; no terminal needed.

# %pip install beautifulsoup4 pandas requests lxml tqdm playwright

## 1. Parameters

Change these depending on what you need.

Important:
- `SITEMAP_LAYERS` controls how deep sitemap recursion goes.
- `MAX_PAGES` controls the maximum number of pages/profile URLs to fetch.
- `MAX_SCROLLS` controls how far Playwright scrolls on dynamic pages.

In [ ]:
from urllib.parse import urljoin, urlparse
import re
import time
import json
import html
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from xml.etree import ElementTree as ET

# =========================
# MAIN PARAMETERS
# =========================

# Paste the main website or event URL here.
# For Swapcard-style pages, this is often the rendered page URL.
WEBSITE_URL = "https://www.hardwarepioneers.com/"

# Base URL used to convert relative URLs such as /widget/event/... into full URLs.
# If unsure, use the domain you copied the HTML from.
BASE_URL = WEBSITE_URL

# Use pasted HTML first. This is best when Inspect Element shows links but View Source does not.
USE_PASTED_HTML = True

# Crawl website/sitemaps too?
USE_SITEMAP_CRAWL = True

# How many sitemap index layers to follow.
# 0 = only the exact sitemap URL
# 1 = sitemap + child sitemaps
# 2 = sitemap + child + grandchild sitemaps
SITEMAP_LAYERS = 2

# Hard cap on how many pages/profile URLs to fetch/enrich.
MAX_PAGES = 300

# If using Playwright rendering, how many times to scroll down.
MAX_SCROLLS = 20

# Delay between requests to avoid hammering the website.
REQUEST_DELAY_SECONDS = 0.2

# Try to enrich company profiles by visiting exhibitor URLs and scraping profile text.
# This improves ranking because company names alone are not enough.
ENRICH_PROFILE_PAGES = True

# If True, only keep URLs likely to be exhibitor/company pages.
FILTER_TO_EXHIBITORS = True

# Keywords that identify exhibitor URLs.
EXHIBITOR_URL_PATTERNS = [
    "/exhibitor/",
    "exhibitor",
    "company",
    "companies",
    "sponsor",
    "sponsors"
]

## 2. Paste your HTML here

Paste the HTML you copied from Inspect Element into the triple quotes.

Tip:
- If you copied a whole exhibitor grid, this is usually enough.
- You do **not** need View Source.

In [ ]:
html_text = r"""
PASTE_YOUR_INSPECT_ELEMENT_HTML_HERE
"""

## 3. Helper functions

In [ ]:
def safe_text(x):
    if x is None:
        return ""
    return str(x).strip()

def normalise_url(href, base_url=BASE_URL):
    href = safe_text(href)
    if not href:
        return ""
    if href.startswith(("mailto:", "tel:", "javascript:", "#")):
        return ""
    return urljoin(base_url, href)

def is_likely_exhibitor_url(url):
    u = url.lower()
    return any(pattern.lower() in u for pattern in EXHIBITOR_URL_PATTERNS)

def clean_whitespace(text):
    return re.sub(r"\s+", " ", safe_text(text)).strip()

def request_get(url, timeout=20):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        if r.status_code == 200:
            return r.text
        return ""
    except Exception:
        return ""

def extract_links_from_html(raw_html, base_url=BASE_URL):
    soup = BeautifulSoup(raw_html or "", "html.parser")
    rows = []

    for a in soup.select("a[href]"):
        href = a.get("href", "")
        url = normalise_url(href, base_url)
        if not url:
            continue

        if FILTER_TO_EXHIBITORS and not is_likely_exhibitor_url(url):
            continue

        img = a.find("img")
        img_alt = clean_whitespace(img.get("alt", "")) if img else ""

        card_text = clean_whitespace(a.get_text(" ", strip=True))
        spans = [clean_whitespace(s.get_text(" ", strip=True)) for s in a.find_all("span")]
        spans = [s for s in spans if s]

        # Common Swapcard pattern:
        # span[0] = company
        # span[1] = stand/sponsor/startup zone
        company_from_span = spans[0] if len(spans) >= 1 else ""
        stand = spans[1] if len(spans) >= 2 else ""

        company = img_alt or company_from_span

        rows.append({
            "company": company,
            "stand": stand,
            "url": url,
            "card_text": card_text,
            "source": "pasted_html"
        })

    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame(columns=["company", "stand", "url", "card_text", "source"])

    df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    return df

def extract_page_text(raw_html):
    soup = BeautifulSoup(raw_html or "", "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.extract()

    title = clean_whitespace(soup.title.get_text(" ", strip=True)) if soup.title else ""
    meta_desc = ""
    meta = soup.find("meta", attrs={"name": "description"})
    if meta:
        meta_desc = clean_whitespace(meta.get("content", ""))

    body_text = clean_whitespace(soup.get_text(" ", strip=True))

    return clean_whitespace(" ".join([title, meta_desc, body_text]))

## 4. Extract links from pasted HTML

This is the fastest and best method for dynamic event websites.

In [ ]:
if USE_PASTED_HTML:
    pasted_df = extract_links_from_html(html_text, BASE_URL)
else:
    pasted_df = pd.DataFrame(columns=["company", "stand", "url", "card_text", "source"])

print(f"Extracted from pasted HTML: {len(pasted_df)} rows")
pasted_df.head(20)

## 5. Optional: Sitemap crawler with `SITEMAP_LAYERS` and `MAX_PAGES`

This tries common sitemap URLs and follows sitemap indexes.

Use this as backup. For JavaScript-heavy event platforms, pasted rendered HTML usually works better.

In [ ]:
def guess_sitemap_urls(site_url):
    parsed = urlparse(site_url)
    root = f"{parsed.scheme}://{parsed.netloc}"
    return [
        urljoin(root, "/sitemap.xml"),
        urljoin(root, "/sitemap_index.xml"),
        urljoin(root, "/sitemap-index.xml"),
        urljoin(root, "/robots.txt"),
    ]

def parse_robots_for_sitemaps(robots_text):
    sitemaps = []
    for line in robots_text.splitlines():
        if line.lower().startswith("sitemap:"):
            sitemaps.append(line.split(":", 1)[1].strip())
    return sitemaps

def parse_sitemap_xml(xml_text):
    urls = []
    child_sitemaps = []

    if not xml_text.strip():
        return urls, child_sitemaps

    try:
        root = ET.fromstring(xml_text.encode("utf-8"))
    except Exception:
        return urls, child_sitemaps

    # Namespace-safe ending checks
    for elem in root.iter():
        tag = elem.tag.lower()
        if tag.endswith("loc") and elem.text:
            loc = elem.text.strip()
            if loc.lower().endswith(".xml") or "sitemap" in loc.lower():
                child_sitemaps.append(loc)
            else:
                urls.append(loc)

    return urls, child_sitemaps

def collect_urls_from_sitemaps(site_url, sitemap_layers=2, max_pages=300):
    initial = guess_sitemap_urls(site_url)

    sitemap_queue = []
    seen_sitemaps = set()
    page_urls = []

    for u in initial:
        txt = request_get(u)
        time.sleep(REQUEST_DELAY_SECONDS)

        if not txt:
            continue

        if u.endswith("robots.txt"):
            sitemap_queue.extend(parse_robots_for_sitemaps(txt))
        else:
            sitemap_queue.append(u)

    # fallback: try standard sitemap if robots didn't give anything
    if not sitemap_queue:
        sitemap_queue = [urljoin(site_url, "/sitemap.xml")]

    current_layer = 0

    while sitemap_queue and current_layer <= sitemap_layers and len(page_urls) < max_pages:
        next_queue = []

        for sitemap_url in list(dict.fromkeys(sitemap_queue)):
            if sitemap_url in seen_sitemaps:
                continue

            seen_sitemaps.add(sitemap_url)
            xml_text = request_get(sitemap_url)
            time.sleep(REQUEST_DELAY_SECONDS)

            urls, child_sitemaps = parse_sitemap_xml(xml_text)

            for u in urls:
                if len(page_urls) >= max_pages:
                    break
                page_urls.append(u)

            next_queue.extend(child_sitemaps)

        sitemap_queue = next_queue
        current_layer += 1

    page_urls = list(dict.fromkeys(page_urls))[:max_pages]
    return page_urls

if USE_SITEMAP_CRAWL:
    sitemap_urls = collect_urls_from_sitemaps(
        WEBSITE_URL,
        sitemap_layers=SITEMAP_LAYERS,
        max_pages=MAX_PAGES
    )
else:
    sitemap_urls = []

print(f"Collected from sitemap crawl: {len(sitemap_urls)} URLs")
sitemap_urls[:20]

## 6. Optional: Extract exhibitor/company links from crawled pages

This visits pages found via sitemap and extracts links from their HTML.

Cap is controlled by `MAX_PAGES`.

In [ ]:
def extract_links_from_urls(urls, max_pages=MAX_PAGES):
    frames = []
    urls = urls[:max_pages]

    for url in tqdm(urls, desc="Scanning sitemap pages"):
        raw = request_get(url)
        time.sleep(REQUEST_DELAY_SECONDS)

        if not raw:
            continue

        tmp = extract_links_from_html(raw, url)
        if not tmp.empty:
            tmp["source"] = "sitemap_page"
            tmp["source_page"] = url
            frames.append(tmp)

    if not frames:
        return pd.DataFrame(columns=["company", "stand", "url", "card_text", "source", "source_page"])

    return pd.concat(frames, ignore_index=True).drop_duplicates(subset=["url"]).reset_index(drop=True)

if USE_SITEMAP_CRAWL and sitemap_urls:
    sitemap_links_df = extract_links_from_urls(sitemap_urls, MAX_PAGES)
else:
    sitemap_links_df = pd.DataFrame(columns=["company", "stand", "url", "card_text", "source", "source_page"])

print(f"Extracted exhibitor/company links from sitemap pages: {len(sitemap_links_df)}")
sitemap_links_df.head(20)

## 7. Combine all extracted links

In [ ]:
all_links_df = pd.concat([pasted_df, sitemap_links_df], ignore_index=True)

if all_links_df.empty:
    all_links_df = pd.DataFrame(columns=["company", "stand", "url", "card_text", "source"])

all_links_df = all_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

# Try to repair missing company names from URL if needed
def company_from_url(url):
    last = url.rstrip("/").split("/")[-1]
    last = re.sub(r"[-_]+", " ", last)
    last = re.sub(r"\s+", " ", last).strip()
    return last

mask_missing = all_links_df["company"].fillna("").str.strip().eq("")
all_links_df.loc[mask_missing, "company"] = all_links_df.loc[mask_missing, "url"].apply(company_from_url)

print(f"Total unique extracted company/exhibitor URLs: {len(all_links_df)}")
all_links_df.head(30)

## 8. Optional: enrich profile pages

This visits each exhibitor/company profile URL and pulls page text.

This improves ranking a lot, but dynamic pages may return limited text unless the content is server-rendered.

In [ ]:
def enrich_profiles(df, max_pages=MAX_PAGES):
    df = df.copy()

    profile_texts = []
    fetched_ok = []

    urls = df["url"].fillna("").tolist()[:max_pages]
    url_to_text = {}

    for url in tqdm(urls, desc="Enriching profile pages"):
        raw = request_get(url)
        time.sleep(REQUEST_DELAY_SECONDS)

        text = extract_page_text(raw) if raw else ""
        url_to_text[url] = text

    df["profile_text"] = df["url"].map(url_to_text).fillna("")
    df["fetched_profile_text"] = df["profile_text"].str.len() > 50

    return df

if ENRICH_PROFILE_PAGES and not all_links_df.empty:
    enriched_df = enrich_profiles(all_links_df, MAX_PAGES)
else:
    enriched_df = all_links_df.copy()
    enriched_df["profile_text"] = ""
    enriched_df["fetched_profile_text"] = False

print(f"Rows after enrichment: {len(enriched_df)}")
enriched_df.head(20)

## 9. Keyword ranking

Edit the weights if you want to bias toward FPGA, embedded, AI, hardware, manufacturing, etc.

Higher score = more relevant.

In [ ]:
KEYWORD_WEIGHTS = {
    # Ultra-relevant to Computer Systems / FPGA / embedded
    "fpga": 15,
    "vhdl": 15,
    "verilog": 15,
    "hdl": 12,
    "asic": 14,
    "soc": 12,
    "rtl": 12,
    "risc-v": 12,
    "risc v": 12,
    "embedded linux": 12,
    "embedded": 11,
    "firmware": 11,
    "microcontroller": 10,
    "mcu": 10,
    "arm": 10,
    "stm32": 10,
    "nordic": 9,
    "esp32": 9,
    "real-time": 9,
    "rtos": 9,

    # Semiconductor / electronics
    "semiconductor": 14,
    "silicon": 12,
    "chip": 9,
    "chips": 9,
    "component": 6,
    "components": 6,
    "electronics": 8,
    "electronic": 8,
    "pcb": 8,
    "pcba": 8,
    "circuit": 7,
    "circuits": 7,
    "sensor": 8,
    "sensors": 8,
    "wireless": 7,
    "rf": 7,
    "antenna": 7,
    "connectors": 5,
    "power": 6,
    "battery": 6,

    # AI / ML / Edge AI
    "edge ai": 12,
    "tinyml": 12,
    "machine learning": 10,
    "artificial intelligence": 8,
    "computer vision": 9,
    "ai": 6,
    "ml": 6,
    "inference": 7,
    "neural": 7,

    # Robotics / automation / systems
    "robotics": 8,
    "robot": 8,
    "automation": 7,
    "iot": 8,
    "iiot": 8,
    "industrial": 5,
    "control": 5,
    "monitoring": 5,

    # Software/dev broad match
    "python": 5,
    "c++": 6,
    "c/c++": 6,
    "linux": 6,
    "software": 5,
    "sdk": 5,
    "api": 4,
    "cloud": 3,

    # Job/career signals
    "careers": 4,
    "graduate": 5,
    "internship": 5,
    "hiring": 5,
    "jobs": 4,
}

NEGATIVE_KEYWORDS = {
    # not necessarily bad, just less aligned for FPGA/embedded career targeting
    "packaging": -1,
    "marketing": -2,
    "press": -1,
}

def keyword_score_text(text):
    text = clean_whitespace(text).lower()

    score = 0
    matches = []

    for kw, weight in KEYWORD_WEIGHTS.items():
        kw_l = kw.lower()

        # Word-ish boundary, but allow symbols like c++
        if re.search(r"(?<![a-z0-9])" + re.escape(kw_l) + r"(?![a-z0-9])", text):
            score += weight
            matches.append(kw)

    for kw, penalty in NEGATIVE_KEYWORDS.items():
        kw_l = kw.lower()
        if re.search(r"(?<![a-z0-9])" + re.escape(kw_l) + r"(?![a-z0-9])", text):
            score += penalty

    return score, ", ".join(matches)

def score_row(row):
    searchable_text = " ".join([
        safe_text(row.get("company", "")),
        safe_text(row.get("stand", "")),
        safe_text(row.get("url", "")),
        safe_text(row.get("card_text", "")),
        safe_text(row.get("profile_text", "")),
    ])

    score, matches = keyword_score_text(searchable_text)

    # Small bonus for explicit startup zone if looking for approachable companies
    stand_text = safe_text(row.get("stand", "")).lower()
    if "startup" in stand_text:
        score += 3
        if matches:
            matches += ", startup zone"
        else:
            matches = "startup zone"

    return pd.Series({
        "keyword_score": score,
        "matched_keywords": matches
    })

ranked_df = enriched_df.copy()

if ranked_df.empty:
    ranked_df["keyword_score"] = []
    ranked_df["matched_keywords"] = []
else:
    scores = ranked_df.apply(score_row, axis=1)
    ranked_df = pd.concat([ranked_df, scores], axis=1)
    ranked_df = ranked_df.sort_values(
        by=["keyword_score", "company"],
        ascending=[False, True]
    ).reset_index(drop=True)

# Nice column order
preferred_cols = [
    "keyword_score", "company", "stand", "matched_keywords",
    "url", "card_text", "fetched_profile_text", "source", "profile_text"
]
existing_cols = [c for c in preferred_cols if c in ranked_df.columns]
other_cols = [c for c in ranked_df.columns if c not in existing_cols]
ranked_df = ranked_df[existing_cols + other_cols]

ranked_df.head(50)

## 10. Quick useful views

In [ ]:
# Top ranked companies
top_ranked = ranked_df[ranked_df["keyword_score"] > 0].head(50)
top_ranked[["keyword_score", "company", "stand", "matched_keywords", "url"]]

In [ ]:
# Companies with no keyword matches
no_match = ranked_df[ranked_df["keyword_score"] <= 0]
print(f"No/low keyword match rows: {len(no_match)}")
no_match[["company", "stand", "url"]].head(30)

In [ ]:
# Search manually inside results
SEARCH_TERM = "semiconductor"  # change this

manual_search = ranked_df[
    ranked_df.astype(str).apply(
        lambda row: row.str.contains(SEARCH_TERM, case=False, na=False).any(),
        axis=1
    )
]

manual_search[["keyword_score", "company", "stand", "matched_keywords", "url"]].head(50)

## 11. Export CSV files

In [ ]:
all_links_df.to_csv("extracted_exhibitor_links.csv", index=False)
ranked_df.to_csv("ranked_exhibitors_by_keywords.csv", index=False)

print("Saved:")
print("- extracted_exhibitor_links.csv")
print("- ranked_exhibitors_by_keywords.csv")

## 12. Emergency browser-console extractor

If Python misses the links but the browser shows them, run this in DevTools Console:

```js
copy(
  [...document.querySelectorAll('a[href]')]
    .filter(a => a.href.toLowerCase().includes('exhibitor'))
    .map(a => {
      const img = a.querySelector('img');
      const spans = [...a.querySelectorAll('span')].map(s => s.innerText.trim()).filter(Boolean);
      return [
        img?.alt || spans[0] || '',
        spans[1] || '',
        a.href
      ].join('|');
    })
    .join('\n')
)
```

Then paste the copied output into a text file or directly into Python.

In [ ]:
# Optional parser for the emergency console output above.
# Paste copied browser console output here if needed.

console_output = r"""
PASTE_BROWSER_CONSOLE_OUTPUT_HERE
"""

def parse_console_pipe_output(text):
    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line or "|" not in line:
            continue

        parts = line.split("|")
        if len(parts) >= 3:
            company = parts[0].strip()
            stand = parts[1].strip()
            url = "|".join(parts[2:]).strip()
            rows.append({
                "company": company,
                "stand": stand,
                "url": url,
                "card_text": f"{company} {stand}",
                "source": "browser_console"
            })

    return pd.DataFrame(rows).drop_duplicates(subset=["url"]).reset_index(drop=True)

console_df = parse_console_pipe_output(console_output)
print(f"Parsed browser console rows: {len(console_df)}")
console_df.head(20)